In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!ls ../input/competitions/birdclef-2026

recording_location.txt	test_soundscapes  train_soundscapes
sample_submission.csv	train_audio	  train_soundscapes_labels.csv
taxonomy.csv		train.csv


In [3]:
BASE = "../input/competitions/birdclef-2026/"

In [4]:
import os
import ast
import torch
import torchaudio
import torchaudio.transforms as T
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

BASE = "../input/competitions/birdclef-2026/"
TRAIN_AUDIO_DIR = os.path.join(BASE, "train_audio")
TRAIN_CSV = os.path.join(BASE, "train.csv")
TAXONOMY_CSV = os.path.join(BASE, "taxonomy.csv")
SOUNDSCAPE_DIR = os.path.join(BASE, "train_soundscapes")
SOUNDSCAPE_CSV = os.path.join(BASE, "train_soundscapes_labels.csv")

class Config:
    SR = 32000               
    DURATION = 5             
    MAX_LENGTH = SR * DURATION 
    N_MELS = 128             
    N_FFT = 1024
    HOP_LENGTH = 512
    BATCH_SIZE = 32
    NUM_WORKERS = 4
    
taxonomy_df = pd.read_csv(TAXONOMY_CSV)
CLASSES = taxonomy_df['primary_label'].unique().tolist()
NUM_CLASSES = len(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

train_df = pd.read_csv(TRAIN_CSV)
soundscape_df = pd.read_csv(SOUNDSCAPE_CSV)

def to_seconds(ts):
    if isinstance(ts, (int, float)): return float(ts)
    parts = str(ts).split(":")
    if len(parts) == 3: return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    if len(parts) == 2: return int(parts[0]) * 60 + float(parts[1])
    return float(ts)

train_rows = []
for _, row in train_df.iterrows():
    labels = [row['primary_label']]
    sec_labels = ast.literal_eval(row.get('secondary_labels', "[]"))
    labels.extend(sec_labels)
    
    train_rows.append({
        'filename': row['filename'],
        'audio_path': os.path.join(TRAIN_AUDIO_DIR, row['filename']),
        'all_labels': list(set(labels)),
        'is_soundscape': 0,
        'start_sec': 0.0
    })

soundscape_rows = []
for _, row in soundscape_df.iterrows():
    labels = [lab for lab in str(row['primary_label']).split(';') if lab in class_to_idx]
    
    soundscape_rows.append({
        'filename': row['filename'],
        'audio_path': os.path.join(SOUNDSCAPE_DIR, row['filename']),
        'all_labels': labels,
        'is_soundscape': 1,
        'start_sec': to_seconds(row['start']) 
    })

clean_manifest = pd.DataFrame(train_rows)
soundscape_manifest = pd.DataFrame(soundscape_rows)

# Oversampling (add soudscapes twice)
train_manifest = pd.concat([clean_manifest, soundscape_manifest, soundscape_manifest], ignore_index=True)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(train_manifest, groups=train_manifest['filename']))

train_split = train_manifest.iloc[train_idx].reset_index(drop=True)
val_split = train_manifest.iloc[val_idx].reset_index(drop=True)

print(f"Train size: {len(train_split)} | Val size: {len(val_split)}")

Train size: 30933 | Val size: 7572


In [5]:
print(f"Кількість файлів у трейні: {len(train_split)}")
print(f"Перший шлях: {train_split.iloc[0]['audio_path']}")
import os
print(f"Файл існує: {os.path.exists(train_split.iloc[0]['audio_path'])}")

Кількість файлів у трейні: 30933
Перший шлях: ../input/competitions/birdclef-2026/train_audio/1161364/iNat1216197.ogg
Файл існує: True


In [6]:
class BirdCLEFDataset(Dataset):
    def __init__(self, df, config, is_train=True):
        self.df = df
        self.config = config
        self.is_train = is_train
        
        self.mel_transform = T.MelSpectrogram(
            sample_rate=config.SR,
            n_fft=config.N_FFT,
            hop_length=config.HOP_LENGTH,
            n_mels=config.N_MELS,
            f_min=50,
            f_max=14000
        )
        self.amp_to_db = T.AmplitudeToDB()
        
        if self.is_train:
            self.time_masking = T.TimeMasking(time_mask_param=30)
            self.freq_masking = T.FrequencyMasking(freq_mask_param=20)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = row['audio_path']
        
        if row['is_soundscape'] == 1:
            start_frame = int(row['start_sec'] * self.config.SR)
            num_frames = self.config.MAX_LENGTH
            waveform, sr = torchaudio.load(audio_path, frame_offset=start_frame, num_frames=num_frames)
        else:
            waveform, sr = torchaudio.load(audio_path)
            
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        audio_len = waveform.shape[1]
        
        if audio_len > self.config.MAX_LENGTH:
            if self.is_train and row['is_soundscape'] == 0:
                max_start = audio_len - self.config.MAX_LENGTH
                start = np.random.randint(0, max_start)
            else:
                start = (audio_len - self.config.MAX_LENGTH) // 2
            waveform = waveform[:, start:start + self.config.MAX_LENGTH]
            
        elif audio_len < self.config.MAX_LENGTH:
            pad_len = self.config.MAX_LENGTH - audio_len
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
            
        mel_spec = self.mel_transform(waveform)
        mel_spec = self.amp_to_db(mel_spec)
        
        # Augmentation
        if self.is_train:
            mel_spec = self.time_masking(mel_spec)
            mel_spec = self.freq_masking(mel_spec)
            
            noise = torch.randn_like(mel_spec) * 0.1 * (mel_spec.max() - mel_spec.min())
            mel_spec = mel_spec + noise
        
        mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
        mel_spec = mel_spec * 2 - 1
        mel_spec = mel_spec.expand(3, -1, -1)
        
        target = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        for label in row['all_labels']:
            if label in class_to_idx:
                target[class_to_idx[label]] = 1.0

        return mel_spec, target

In [7]:
train_dataset = BirdCLEFDataset(train_split, Config, is_train=True)
val_dataset = BirdCLEFDataset(val_split, Config, is_train=False)

train_loader = DataLoader(
    train_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=Config.NUM_WORKERS,
    pin_memory=True
)

In [8]:
images, targets = next(iter(train_loader))
print(f"Розмір картинок: {images.shape}") # Має бути [32, 3, 128, 313]
print(f"Розмір таргетів: {targets.shape}") # Має бути [32, 234]

Розмір картинок: torch.Size([32, 3, 128, 313])
Розмір таргетів: torch.Size([32, 234])


In [9]:
import torch.nn as nn
import timm

class BirdCLEFModel(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=234, pretrained=True):
        super().__init__()
        
        self.backbone = timm.create_model(
            model_name, 
            pretrained=pretrained, 
            num_classes=0 
        )
        
        in_features = self.backbone.num_features
        
        self.head = nn.Linear(in_features, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        
        logits = self.head(features)
        
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BirdCLEFModel().to(device)
print(f"Model loaded on {device}")


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model loaded on cuda


In [10]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

model.train()

images, targets = next(iter(train_loader))
images = images.to(device)
targets = targets.to(device)

optimizer.zero_grad()
outputs = model(images)

loss = criterion(outputs, targets)

loss.backward()
optimizer.step()

print(f"Success! Forward and backward pass complete. Loss: {loss.item():.4f}")

Success! Forward and backward pass complete. Loss: 0.6974


In [11]:
model.eval()
with torch.no_grad():
    output = model(images.to(device))
    print(f"Вихід моделі: {output.shape}") # Має бути [32, 234]

Вихід моделі: torch.Size([32, 234])


In [12]:
import glob
import os
import torch
import torchaudio
import pandas as pd

TEST_AUDIO_DIR = os.path.join(BASE, "test_soundscapes")
test_files = glob.glob(os.path.join(TEST_AUDIO_DIR, "*.ogg"))

if len(test_files) == 0:
    print("Test directory empty. Using train_soundscapes for pipeline test...")
    TEST_AUDIO_DIR = os.path.join(BASE, "train_soundscapes")
    test_files = glob.glob(os.path.join(TEST_AUDIO_DIR, "*.ogg"))[:2] 

model.eval()
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=Config.SR, 
    n_fft=Config.N_FFT, 
    hop_length=Config.HOP_LENGTH, 
    n_mels=Config.N_MELS, 
    f_min=50, 
    f_max=14000
).to(device)
amp_to_db = torchaudio.transforms.AmplitudeToDB().to(device)

predictions = []

print(f"Processing {len(test_files)} files...")
with torch.no_grad():
    for file_path in test_files:
        filename = os.path.basename(file_path)
        file_id = filename.replace('.ogg', '')
        
        waveform, sr = torchaudio.load(file_path)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        waveform = waveform.to(device)
        
        chunk_length = Config.MAX_LENGTH # 160,000 samples
        num_chunks = waveform.shape[1] // chunk_length
        
        if num_chunks == 0:
            pad_len = chunk_length - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
            num_chunks = 1
            
        for i in range(num_chunks):
            start = i * chunk_length
            end = start + chunk_length
            chunk = waveform[:, start:end]
            
            mel_spec = mel_transform(chunk)
            mel_spec = amp_to_db(mel_spec)
            mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
            mel_spec = mel_spec * 2 - 1
            
            mel_spec = mel_spec.unsqueeze(0).expand(-1, 3, -1, -1)
            
            logits = model(mel_spec)
            probs = torch.sigmoid(logits).cpu().numpy()[0]
            
            end_time = (i + 1) * 5 # e.g., 5, 10, 15... 60
            row_id = f"{file_id}_{end_time}"
            
            pred_dict = {'row_id': row_id}
            for class_name, prob in zip(CLASSES, probs):
                pred_dict[class_name] = prob
                
            predictions.append(pred_dict)

submission_df = pd.DataFrame(predictions)
submission_df.to_csv('submission.csv', index=False)
print("submission.csv successfully created!")

Test directory empty. Using train_soundscapes for pipeline test...
Processing 2 files...
submission.csv successfully created!


In [13]:
import time
import numpy as np
import torch
import pandas as pd
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

def calculate_competition_roc_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) == 2:
            class_auc = roc_auc_score(y_true[:, i], y_pred[:, i])
            aucs.append(class_auc)
    if len(aucs) == 0:
        return 0.5
    return np.mean(aucs)

EPOCHS = 10
best_val_auc = 0.0

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

history = []
WORK_DIR = '/kaggle/working/'

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    
    for images, targets in train_pbar:
        images = images.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        
        logits = model(images)
        loss = criterion(logits, targets)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        train_pbar.set_postfix(batch_loss=f"{loss.item():.4f}")
        
    train_loss = train_loss / len(train_loader.dataset)
    
    model.eval()
    val_loss = 0.0
    all_val_targets = []
    all_val_preds = []
    
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False)
    
    with torch.no_grad():
        for images, targets in val_pbar:
            images = images.to(device)
            targets = targets.to(device)
            
            logits = model(images)
            loss = criterion(logits, targets)
            val_loss += loss.item() * images.size(0)
            
            probs = torch.sigmoid(logits)
            all_val_targets.append(targets.cpu().numpy())
            all_val_preds.append(probs.cpu().numpy())
            
            val_pbar.set_postfix(batch_loss=f"{loss.item():.4f}")
            
    val_loss = val_loss / len(val_loader.dataset)
    all_val_targets = np.vstack(all_val_targets)
    all_val_preds = np.vstack(all_val_preds)
    
    val_auc = calculate_competition_roc_auc(all_val_targets, all_val_preds)
    
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    
    print(f"Epoch {epoch+1}/{EPOCHS} | LR: {current_lr:.2e}")
    print(f"  -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val ROC-AUC: {val_auc:.4f}")
    
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_auc': val_auc,
        'lr': current_lr
    })
    
    pd.DataFrame(history).to_csv(f"{WORK_DIR}training_history.csv", index=False)
    
    if val_auc > best_val_auc:
        print(f"  [+] Validation AUC improved ({best_val_auc:.4f} -> {val_auc:.4f}). Saving best model!")
        best_val_auc = val_auc
        torch.save(model.state_dict(), f"{WORK_DIR}best_birdclef_model.pth")

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_auc': val_auc
    }, f"{WORK_DIR}last_checkpoint.pth")

print("Training Complete! All artifacts saved to /kaggle/working/")

Epoch 1/10 | LR: 1.00e-03
  -> Train Loss: 0.0325 | Val Loss: 0.0752 | Val ROC-AUC: 0.6787
  [+] Validation AUC improved (0.0000 -> 0.6787). Saving best model!


Epoch 2/10 | LR: 9.76e-04
  -> Train Loss: 0.0241 | Val Loss: 0.0292 | Val ROC-AUC: 0.8296
  [+] Validation AUC improved (0.6787 -> 0.8296). Saving best model!


Epoch 3/10 | LR: 9.05e-04
  -> Train Loss: 0.0205 | Val Loss: 0.0256 | Val ROC-AUC: 0.8777
  [+] Validation AUC improved (0.8296 -> 0.8777). Saving best model!


Epoch 4/10 | LR: 7.94e-04
  -> Train Loss: 0.0184 | Val Loss: 0.0234 | Val ROC-AUC: 0.8914
  [+] Validation AUC improved (0.8777 -> 0.8914). Saving best model!


Epoch 5/10 | LR: 6.55e-04
  -> Train Loss: 0.0169 | Val Loss: 0.0231 | Val ROC-AUC: 0.8907


Epoch 6/10 | LR: 5.01e-04
  -> Train Loss: 0.0157 | Val Loss: 0.0218 | Val ROC-AUC: 0.8956
  [+] Validation AUC improved (0.8914 -> 0.8956). Saving best model!


Epoch 7/10 | LR: 3.46e-04
  -> Train Loss: 0.0146 | Val Loss: 0.0224 | Val ROC-AUC: 0.8960
  [+] Validation AUC improved (0.8956 -> 0.8960). Saving best model!


Epoch 8/10 | LR: 2.07e-04
  -> Train Loss: 0.0137 | Val Loss: 0.0208 | Val ROC-AUC: 0.9086
  [+] Validation AUC improved (0.8960 -> 0.9086). Saving best model!


Epoch 9/10 | LR: 9.64e-05
  -> Train Loss: 0.0129 | Val Loss: 0.0208 | Val ROC-AUC: 0.9062


Epoch 10/10 | LR: 2.54e-05
  -> Train Loss: 0.0125 | Val Loss: 0.0216 | Val ROC-AUC: 0.9036
Training Complete! All artifacts saved to /kaggle/working/


In [14]:
model.load_state_dict(torch.load('best_birdclef_model.pth'))
model.to(device)
model.eval()

BirdCLEFModel(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_pw):